# Ch06 — 點估計與信賴區間

**Point Estimation & Confidence Intervals**

| 項目 | 說明 |
| :--- | :--- |
| 預計時長 | 1.5 小時 |
| 前置章節 | [Ch05 — 抽樣分佈與中央極限定理](05_抽樣分佈與中央極限定理.ipynb) |
| 資料集 | `datasets/titanic_train.csv` |

## 學習目標

完成本章後，你將能夠：

1. 理解**點估計 (Point Estimation)** 的概念，以及好的估計量應具備的三個性質
2. 正確解讀**信賴區間 (Confidence Interval, CI)** 的統計意義
3. 區分 **Z-CI** 與 **t-CI** 的使用時機
4. 計算**母體比例 (Proportion)** 的信賴區間
5. 理解**樣本大小 (Sample Size)** 對信賴區間寬度的影響

## Import 套件與資料

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)

# 設定隨機種子以確保結果可重現
np.random.seed(42)

In [ ]:
# 載入 Titanic 資料集
titanic = pd.read_csv('datasets/titanic_train.csv')
print(f'資料筆數: {len(titanic)}')
titanic.head()

In [ ]:
# 取出 Age 欄位（移除缺失值），作為本章的「母體」
age_population = titanic['Age'].dropna().values
print(f'Age 有效筆數: {len(age_population)}')
print(f'母體平均 (mu): {age_population.mean():.4f}')
print(f'母體標準差 (sigma): {age_population.std():.4f}')

---
## Part A — 點估計 (Point Estimation)

### 什麼是點估計？

**點估計 (Point Estimation)** 是用樣本統計量 (Statistic) 來估計母體參數 (Parameter) 的方法。

| 母體參數 | 符號 | 對應的點估計量 | 符號 |
| :--- | :---: | :--- | :---: |
| 母體平均數 (Population Mean) | $\mu$ | 樣本平均數 (Sample Mean) | $\bar{x}$ |
| 母體變異數 (Population Variance) | $\sigma^2$ | 樣本變異數 (Sample Variance) | $s^2$ |
| 母體比例 (Population Proportion) | $p$ | 樣本比例 (Sample Proportion) | $\hat{p}$ |

### 好的估計量的三個性質

**1. 不偏性 (Unbiasedness)**
- 估計量的期望值等於母體參數：$E(\hat{\theta}) = \theta$
- 例如：$E(\bar{X}) = \mu$，所以 $\bar{X}$ 是 $\mu$ 的不偏估計量

**2. 一致性 (Consistency)**
- 當樣本數 $n \to \infty$ 時，估計量會收斂到真實參數值
- 隨著 $n$ 增大，估計量的變異數趨近於 0

**3. 有效性 (Efficiency)**
- 在所有不偏估計量中，有效估計量具有最小的變異數
- 變異數越小，估計越精確

### 實作：樣本平均 vs 全體平均

In [ ]:
# 從「母體」(Titanic Age) 中隨機抽樣
mu = age_population.mean()

sample_sizes = [10, 30, 50, 100, 200]
n_simulations = 1000

print(f'母體平均 mu = {mu:.4f}')
print(f'{'─' * 50}')

for n in sample_sizes:
    sample_means = [
        np.random.choice(age_population, size=n, replace=True).mean()
        for _ in range(n_simulations)
    ]
    avg_of_means = np.mean(sample_means)
    std_of_means = np.std(sample_means)
    print(
        f'n = {n:>3d} | '
        f'x-bar 平均 = {avg_of_means:.4f} | '
        f'x-bar 標準差 = {std_of_means:.4f} | '
        f'偏差 = {avg_of_means - mu:+.4f}'
    )

In [ ]:
# 視覺化：不同樣本大小下，樣本平均的分佈
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, n in zip(axes, [10, 50, 200]):
    sample_means = [
        np.random.choice(age_population, size=n, replace=True).mean()
        for _ in range(n_simulations)
    ]
    ax.hist(sample_means, bins=30, edgecolor='black', alpha=0.7, density=True)
    ax.axvline(mu, color='red', linewidth=2, linestyle='--', label=f'$\\mu$ = {mu:.2f}')
    ax.axvline(np.mean(sample_means), color='blue', linewidth=2,
               linestyle=':', label=f'$\\bar{{x}}$ avg = {np.mean(sample_means):.2f}')
    ax.set_title(f'n = {n}', fontsize=14)
    ax.set_xlabel('Sample Mean')
    ax.legend(fontsize=10)

fig.suptitle('Unbiasedness: Sample Mean is an Unbiased Estimator of Population Mean',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

> **觀察**：無論 $n$ 多大或多小，$\bar{X}$ 的分佈都以 $\mu$ 為中心（不偏性）。
> 但 $n$ 越大，分佈越集中（一致性 + 有效性提升）。

---
## Part B — 信賴區間 (Confidence Interval)

### 信賴區間公式

$$\text{CI} = \text{Point Estimate} \pm \text{Critical Value} \times \text{Standard Error}$$

$$\bar{x} \pm z_{\alpha/2} \times \frac{\sigma}{\sqrt{n}}$$

其中：
- $\bar{x}$：樣本平均（點估計）
- $z_{\alpha/2}$：臨界值 (Critical Value)
- $\sigma / \sqrt{n}$：標準誤差 (Standard Error, SE)

### 正確解讀信賴區間

**常見的錯誤解讀：**
> 「真實的母體參數有 95% 的機率落在這個信賴區間內。」

**正確的解讀：**
> 「若我們以相同的方法重複抽樣很多次，每次都計算一個 95% 信賴區間，
> 則大約有 95% 的信賴區間會包含真實的母體參數。」

母體參數 $\mu$ 是一個**固定的未知常數**，不是隨機變數。
信賴區間才是隨機的 — 每次抽樣都會產生不同的區間。

### 圖解：100 個信賴區間的模擬

In [ ]:
# 模擬 100 次抽樣，畫出 100 個 95% CI
n_samples = 100
sample_size = 30
confidence_level = 0.95
z_critical = stats.norm.ppf(1 - (1 - confidence_level) / 2)
sigma = age_population.std()

# 儲存每個 CI
ci_results = []
for i in range(n_samples):
    sample = np.random.choice(age_population, size=sample_size, replace=True)
    x_bar = sample.mean()
    se = sigma / np.sqrt(sample_size)
    ci_lower = x_bar - z_critical * se
    ci_upper = x_bar + z_critical * se
    contains_mu = ci_lower <= mu <= ci_upper
    ci_results.append((x_bar, ci_lower, ci_upper, contains_mu))

In [ ]:
# 視覺化 100 個 CI
fig, ax = plt.subplots(figsize=(10, 14))

for i, (x_bar, ci_lo, ci_hi, contains) in enumerate(ci_results):
    color = 'steelblue' if contains else 'red'
    ax.plot([ci_lo, ci_hi], [i, i], color=color, linewidth=1.5, alpha=0.7)
    ax.plot(x_bar, i, 'o', color=color, markersize=3)

ax.axvline(mu, color='darkred', linewidth=2, linestyle='--',
           label=f'True $\\mu$ = {mu:.2f}')

n_contains = sum(r[3] for r in ci_results)
ax.set_title(
    f'100 Simulated 95% Confidence Intervals\n'
    f'{n_contains}/100 intervals contain the true $\\mu$',
    fontsize=14
)
ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('Sample Index', fontsize=12)
ax.legend(fontsize=12, loc='upper right')
plt.tight_layout()
plt.show()

print(f'包含 mu 的區間比例: {n_contains}/{n_samples} = {n_contains/n_samples:.0%}')

> **觀察**：紅色線段代表**未包含**真實 $\mu$ 的 CI。
> 在 100 個 95% CI 中，大約有 5 個會「漏掉」真實的 $\mu$，
> 這正是「95% 信心水準」的意義。

---
## Part C — Z-CI vs t-CI

### 何時用 Z-CI？何時用 t-CI？

| 條件 | 使用 | 公式 |
| :--- | :---: | :--- |
| $\sigma$ 已知 | Z-CI | $\bar{x} \pm z_{\alpha/2} \times \dfrac{\sigma}{\sqrt{n}}$ |
| $\sigma$ 未知（用 $s$ 估計） | t-CI | $\bar{x} \pm t_{\alpha/2,\, df} \times \dfrac{s}{\sqrt{n}}$ |

**實務上**：母體標準差 $\sigma$ 幾乎不會已知，因此**絕大多數情況都使用 t-CI**。

當 $n$ 很大時（如 $n > 30$），$t$ 分佈趨近常態分佈，Z-CI 與 t-CI 結果會非常接近。

### Z-CI 範例（假設 sigma 已知）

In [ ]:
# 從 Titanic Age 抽一組樣本
sample = np.random.choice(age_population, size=40, replace=True)
x_bar = sample.mean()
n = len(sample)

# Z-CI（假設我們知道 sigma）
sigma_known = age_population.std()
z_alpha_half = stats.norm.ppf(0.975)
se_z = sigma_known / np.sqrt(n)

z_ci_lower = x_bar - z_alpha_half * se_z
z_ci_upper = x_bar + z_alpha_half * se_z

print(f'樣本平均 x-bar = {x_bar:.4f}')
print(f'Z 臨界值 = {z_alpha_half:.4f}')
print(f'標準誤差 SE = {se_z:.4f}')
print(f'95% Z-CI: ({z_ci_lower:.4f}, {z_ci_upper:.4f})')

### t-CI 範例（sigma 未知，實務常用）

In [ ]:
# t-CI（使用樣本標準差 s）
s = sample.std(ddof=1)  # 樣本標準差 (ddof=1 for unbiased)
se_t = s / np.sqrt(n)
df = n - 1

# 方法一：手動計算
t_alpha_half = stats.t.ppf(0.975, df=df)
t_ci_lower = x_bar - t_alpha_half * se_t
t_ci_upper = x_bar + t_alpha_half * se_t

print(f'樣本標準差 s = {s:.4f}')
print(f't 臨界值 (df={df}) = {t_alpha_half:.4f}')
print(f'標準誤差 SE = {se_t:.4f}')
print(f'95% t-CI (手動): ({t_ci_lower:.4f}, {t_ci_upper:.4f})')

In [ ]:
# 方法二：使用 scipy.stats.t.interval()
t_ci = stats.t.interval(
    confidence=0.95,
    df=df,
    loc=x_bar,
    scale=se_t
)

print(f'95% t-CI (scipy): ({t_ci[0]:.4f}, {t_ci[1]:.4f})')

In [ ]:
# 比較 Z-CI 與 t-CI
print(f'{'─' * 40}')
print(f'95% Z-CI: ({z_ci_lower:.4f}, {z_ci_upper:.4f})  寬度 = {z_ci_upper - z_ci_lower:.4f}')
print(f'95% t-CI: ({t_ci[0]:.4f}, {t_ci[1]:.4f})  寬度 = {t_ci[1] - t_ci[0]:.4f}')
print(f'{'─' * 40}')
print('t-CI 通常比 Z-CI 略寬，因為 t 分佈的尾部比常態分佈厚。')

### 用完整 Titanic Age 資料計算 95% CI

In [ ]:
# 將整個 Age 欄位（有效值）視為「樣本」來計算 CI
age_data = titanic['Age'].dropna()
n_age = len(age_data)
mean_age = age_data.mean()
se_age = age_data.std(ddof=1) / np.sqrt(n_age)

ci_95 = stats.t.interval(
    confidence=0.95,
    df=n_age - 1,
    loc=mean_age,
    scale=se_age
)

print(f'Titanic 乘客年齡 (n={n_age})')
print(f'  樣本平均: {mean_age:.2f} 歲')
print(f'  95% CI:   ({ci_95[0]:.2f}, {ci_95[1]:.2f}) 歲')

---
## Part D — 比例的信賴區間 (Confidence Interval for Proportions)

### Wald Interval

對於母體比例 $p$ 的估計，最常見的方法是 **Wald interval**：

$$\hat{p} \pm z_{\alpha/2} \times \sqrt{\frac{\hat{p}(1 - \hat{p})}{n}}$$

其中 $\hat{p}$ 是樣本比例。

In [ ]:
# Titanic 存活率的 95% CI (Wald Interval)
n_total = len(titanic)
n_survived = titanic['Survived'].sum()
p_hat = n_survived / n_total

z_crit = stats.norm.ppf(0.975)
se_prop = np.sqrt(p_hat * (1 - p_hat) / n_total)

wald_lower = p_hat - z_crit * se_prop
wald_upper = p_hat + z_crit * se_prop

print(f'Titanic 存活率')
print(f'  n = {n_total}, 存活 = {n_survived}')
print(f'  p-hat = {p_hat:.4f} ({p_hat:.1%})')
print(f'  95% Wald CI: ({wald_lower:.4f}, {wald_upper:.4f})')
print(f'  即 ({wald_lower:.1%}, {wald_upper:.1%})')

### 知識補給站：Wilson Interval

Wald interval 在 $\hat{p}$ 接近 0 或 1，或 $n$ 較小時，表現不佳（覆蓋率可能遠低於名目水準）。

**Wilson interval** 是一個更穩健的替代方案：

$$\frac{\hat{p} + \frac{z^2}{2n} \pm z \sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$

在 `statsmodels` 中可以用 `proportion_confint` 來計算。

In [ ]:
from statsmodels.stats.proportion import proportion_confint

# Wilson interval
wilson_ci = proportion_confint(
    count=n_survived,
    nobs=n_total,
    alpha=0.05,
    method='wilson'
)

print(f'95% Wald CI:   ({wald_lower:.4f}, {wald_upper:.4f})')
print(f'95% Wilson CI: ({wilson_ci[0]:.4f}, {wilson_ci[1]:.4f})')
print(f'\n兩者在 n 夠大時非常接近，但 Wilson interval 在小樣本時更可靠。')

---
## Part E — 樣本大小的影響 (Effect of Sample Size)

### 核心觀念

$$SE = \frac{s}{\sqrt{n}}$$

- $n \uparrow \Rightarrow SE \downarrow \Rightarrow$ CI 變窄
- 要讓 CI 寬度減半，需要 $4$ 倍的樣本數

In [ ]:
# 不同 n 下，CI 寬度的變化
sample_sizes = np.arange(10, 501, 10)
ci_widths = []
ci_lower_list = []
ci_upper_list = []

s_est = age_population.std(ddof=1)  # 假設固定的樣本標準差估計

for n in sample_sizes:
    se = s_est / np.sqrt(n)
    t_crit = stats.t.ppf(0.975, df=n - 1)
    margin = t_crit * se
    ci_widths.append(2 * margin)
    ci_lower_list.append(mu - margin)
    ci_upper_list.append(mu + margin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：CI 寬度 vs n
axes[0].plot(sample_sizes, ci_widths, color='steelblue', linewidth=2)
axes[0].set_xlabel('Sample Size (n)', fontsize=12)
axes[0].set_ylabel('CI Width', fontsize=12)
axes[0].set_title('95% CI Width vs Sample Size', fontsize=14)
axes[0].grid(True, alpha=0.3)

# 右圖：CI 範圍 vs n
axes[1].fill_between(sample_sizes, ci_lower_list, ci_upper_list,
                     alpha=0.3, color='steelblue', label='95% CI')
axes[1].axhline(mu, color='red', linewidth=2, linestyle='--',
                label=f'$\\mu$ = {mu:.2f}')
axes[1].set_xlabel('Sample Size (n)', fontsize=12)
axes[1].set_ylabel('Age', fontsize=12)
axes[1].set_title('95% CI Shrinks as n Increases', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 數值比較：n 翻倍 vs CI 寬度
print(f'{"n":>6s}  {"CI Width":>10s}  {"Width Ratio (vs n=10)":>22s}')
print(f'{'─' * 42}')

reference_width = None
for n in [10, 20, 40, 80, 160, 320]:
    se = s_est / np.sqrt(n)
    t_crit = stats.t.ppf(0.975, df=n - 1)
    width = 2 * t_crit * se
    if reference_width is None:
        reference_width = width
    ratio = width / reference_width
    print(f'{n:>6d}  {width:>10.4f}  {ratio:>22.2%}')

### 實務思考：「要多少樣本才夠？」

要達到指定的誤差範圍 $E$（CI 的半寬），所需的樣本數為：

$$n = \left( \frac{z_{\alpha/2} \cdot \sigma}{E} \right)^2$$

這個問題將在 **Ch07 — 假設檢定框架** 中進一步討論 **power analysis** 的概念。

In [ ]:
# 範例：若希望 95% CI 的誤差範圍不超過 2 歲，需要多少樣本？
E_target = 2.0  # 歲
z_crit = stats.norm.ppf(0.975)
sigma_est = age_population.std()

n_required = (z_crit * sigma_est / E_target) ** 2
n_required_ceil = int(np.ceil(n_required))

print(f'若希望 95% CI 的誤差範圍 <= {E_target} 歲')
print(f'  估計 sigma = {sigma_est:.4f}')
print(f'  所需樣本數 n >= {n_required:.1f} => 至少 {n_required_ceil} 人')

---
## 重點整理

| 概念 | 要點 |
| :--- | :--- |
| 點估計 (Point Estimation) | 用樣本統計量估計母體參數；$\bar{x}$ 是 $\mu$ 的不偏估計量 |
| 好的估計量性質 | 不偏性 (Unbiased)、一致性 (Consistent)、有效性 (Efficient) |
| 信賴區間 (CI) | CI = 點估計 $\pm$ 臨界值 $\times$ SE |
| CI 正確解讀 | 反覆抽樣下，有 95% 的 CI 會包含真實 $\mu$ |
| Z-CI vs t-CI | $\sigma$ 已知用 Z，$\sigma$ 未知用 t（實務幾乎都用 t） |
| 比例的 CI | Wald interval：$\hat{p} \pm z \sqrt{\hat{p}(1-\hat{p})/n}$ |
| 樣本大小影響 | $n \uparrow \Rightarrow SE \downarrow \Rightarrow$ CI 變窄 |

---
## 練習題

### 練習 1 (Basic)

從 Titanic 的 `Fare` 欄位中隨機抽取 50 筆資料，計算票價的 95% t-CI。

In [ ]:
# 練習 1：你的程式碼


### 練習 2 (Intermediate)

計算 Titanic **各艙等 (Pclass)** 的存活率 95% CI（使用 Wilson interval），
並用 error bar 圖呈現結果。

In [ ]:
# 練習 2：你的程式碼


### 練習 3 (Advanced)

設計一個模擬實驗：
1. 分別以 90%、95%、99% 信心水準，各模擬 500 個 CI
2. 計算每個信心水準下，CI 實際包含 $\mu$ 的比例
3. 將三組結果繪製成長條圖，並與名目覆蓋率比較

In [ ]:
# 練習 3：你的程式碼


---

$\leftarrow$ [Ch05 — 抽樣分佈與中央極限定理](05_抽樣分佈與中央極限定理.ipynb) | [Ch07 — 假設檢定框架](07_假設檢定框架.ipynb) $\rightarrow$